# Phase 3: Model Training and Evaluation

**Goal**: Train ML models with proper train/test split and class imbalance handling

**Input**:
- `data/processed/Phase 2 - Feature Engineering/training_features.csv` (373,114 events, 288 suspicious)
- `data/processed/Phase 2 - Feature Engineering/validation/[dataset]_features.csv` (5 files for final validation)

**Output**:
- `models/best_model.pkl` - Trained model for production
- `data/processed/Phase 3 - Model Training/model_comparison.csv` - Performance comparison
- `data/processed/Phase 3 - Model Training/feature_importance.csv` - Feature rankings
- `data/processed/Phase 3 - Model Training/test_results.csv` - Test set predictions
- `data/processed/Phase 3 - Model Training/validation_predictions.csv` - Validation predictions

**Models**:
1. Random Forest (ensemble baseline)
2. XGBoost (gradient boosting)
3. LightGBM (fast gradient boosting)
4. Logistic Regression (linear baseline)

**Strategy**:
- Case-based split: 18 datasets for training, 4 datasets for testing (82/18 split)
- SMOTE for class imbalance (oversample minority class)
- Class weights for cost-sensitive learning
- Hyperparameter tuning for each model

---

## Setup and Configuration

In [303]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    precision_recall_curve, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import StandardScaler

# Imbalanced learning
from imblearn.over_sampling import SMOTE

# Gradient boosting
import xgboost as xgb
import lightgbm as lgb

print("Phase 3: Model Training and Evaluation")
print("="*80)
print("\nLibraries imported successfully")


Phase 3: Model Training and Evaluation

Libraries imported successfully


## Directory Configuration

**Input**: Training features from Phase 2  
**Output**: Trained models and performance metrics


In [304]:
# Cell 2: Directory Configuration

BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')

# Input paths
TRAINING_FEATURES = BASE_DIR / 'data/processed/Phase 2 - Feature Engineering/training_features.csv'
VALIDATION_FEATURES_DIR = BASE_DIR / 'data/validation/processed/phase 2'

# Output paths
OUTPUT_DIR = BASE_DIR / 'data/processed/Phase 3 - Model Training'
MODELS_DIR = BASE_DIR / 'models'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Directories configured")
print(f"Training features: {TRAINING_FEATURES}")
print(f"Validation features: {VALIDATION_FEATURES_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Models: {MODELS_DIR}")


Directories configured
Training features: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Feature Engineering/training_features.csv
Validation features: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 2
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training
Models: /Users/soni/Github/Digital-Detectives_Thesis/models


## Data Loading and Preparation

### Step 1: Load Training Features

Load the feature-engineered dataset from Phase 2 (373,114 events with 49 columns).


In [305]:
# Cell 3: Load Training Data

print("\n" + "="*80)
print("LOADING TRAINING DATA")
print("="*80)

# Load training features
df = pd.read_csv(TRAINING_FEATURES, low_memory=False)

print(f"\nLoaded training data:")
print(f"  Shape: {df.shape}")
print(f"  Total events: {len(df):,}")
print(f"  Suspicious events: {df['is_suspicious'].sum():,} ({df['is_suspicious'].mean()*100:.3f}%)")
print(f"  Benign events: {(~df['is_suspicious']).sum():,}")
print(f"\nDatasets: {df['dataset'].nunique()}")
print(f"Datasets list: {sorted(df['dataset'].unique())}")



LOADING TRAINING DATA

Loaded training data:
  Shape: (373114, 49)
  Total events: 373,114
  Suspicious events: 288 (0.077%)
  Benign events: 372,826

Datasets: 22
Datasets list: ['01-APT17', '01-PE', '02-PE', '03-APT21', '03-PE', '04-APT28', '04-PE', '05-APT29', '05-PE', '06-APT30', '06-PE', '07-APT37', '07-PE', '08-APT38', '08-PE', '09-PE', '10-DarkHotel663', '10-PE', '11-DarkHotelbbd', '11-PE', '12-PE', '14-Winnti43b']


In [306]:
# Cell 3: Load Training Data

print("\n" + "="*80)
print("LOADING TRAINING DATA")
print("="*80)

# Load training features
df = pd.read_csv(TRAINING_FEATURES, low_memory=False)

print(f"\nLoaded training data:")
print(f"  Shape: {df.shape}")
print(f"  Total events: {len(df):,}")
print(f"  Suspicious events: {df['is_suspicious'].sum():,} ({df['is_suspicious'].mean()*100:.3f}%)")
print(f"  Benign events: {(~df['is_suspicious']).sum():,}")
print(f"\nDatasets: {df['dataset'].nunique()}")
print(f"Datasets list: {sorted(df['dataset'].unique())}")



LOADING TRAINING DATA

Loaded training data:
  Shape: (373114, 49)
  Total events: 373,114
  Suspicious events: 288 (0.077%)
  Benign events: 372,826

Datasets: 22
Datasets list: ['01-APT17', '01-PE', '02-PE', '03-APT21', '03-PE', '04-APT28', '04-PE', '05-APT29', '05-PE', '06-APT30', '06-PE', '07-APT37', '07-PE', '08-APT38', '08-PE', '09-PE', '10-DarkHotel663', '10-PE', '11-DarkHotelbbd', '11-PE', '12-PE', '14-Winnti43b']


### Step 3: Handle Missing Values

Fill missing values with appropriate defaults:
- Boolean features: False
- Numeric features: 0 or median
- Datetime features: Drop or convert to numeric


In [307]:
# Cell 4: Feature Selection (FIXED)

print("\n" + "="*80)
print("FEATURE SELECTION")
print("="*80)

# Columns to drop (non-predictive)
drop_cols = [
    # Identifiers
    'LSN', 'USN', 'dataset', 'filename', 'full_path', 
    'file_ref_number', 'parent_file_ref_number',
    
    # Raw text fields (categorical, not encoded)
    'detail_lf', 'category', 'detail', 'file_extension', 'source_artifact',
    'file_attribute',  # ADDED: Contains categorical text like "Hidden / System"
    
    # Datetime strings (keep only parsed datetime and numeric features)
    'event_time', 'timestamp_before', 'timestamp_after',
    'CreationTime', 'ModifiedTime', 'MFTModifiedTime', 'AccessedTime',
    
    # Target variable (will use separately)
    'is_suspicious'
]

# Only drop columns that exist
drop_cols = [col for col in drop_cols if col in df.columns]

# Create feature matrix X and target y
X = df.drop(columns=drop_cols)
y = df['is_suspicious'].astype(int)

# Keep dataset info for train/test split
dataset_info = df['dataset'].copy()

print(f"\nOriginal columns: {len(df.columns)}")
print(f"Dropped columns: {len(drop_cols)}")
print(f"Feature columns: {len(X.columns)}")

# Check for any remaining non-numeric columns
non_numeric_cols = X.select_dtypes(exclude=[np.number, 'bool', 'datetime64']).columns
if len(non_numeric_cols) > 0:
    print(f"\nWARNING: Non-numeric columns detected: {list(non_numeric_cols)}")
    print("These will be dropped:")
    X = X.drop(columns=non_numeric_cols)
    print(f"Final feature columns after removing non-numeric: {len(X.columns)}")

print(f"\nFeature columns:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nTarget distribution:")
print(f"  Class 0 (benign): {(y==0).sum():,} ({(y==0).mean()*100:.3f}%)")
print(f"  Class 1 (suspicious): {(y==1).sum():,} ({(y==1).mean()*100:.3f}%)")
print(f"  Class imbalance ratio: 1:{(y==0).sum()/(y==1).sum():.0f}")



FEATURE SELECTION

Original columns: 49
Dropped columns: 21
Feature columns: 28

These will be dropped:
Final feature columns after removing non-numeric: 24

Feature columns:
   1. zero_in_nanoseconds
   2. zero_in_creation
   3. zero_in_modified
   4. zero_in_mft_modified
   5. zero_in_accessed
   6. time_reversal_event
   7. basic_info_changed
   8. update_resident_value
   9. timestamp_to_past
  10. using_another_timestamp
  11. has_logfile_evidence
  12. has_usnjrnl_evidence
  13. cross_artifact_validation_score
  14. event_count_per_file
  15. events_in_1min_window
  16. events_in_5min_window
  17. is_executable
  18. is_document
  19. is_image
  20. is_archive
  21. is_system_file
  22. path_depth
  23. filename_length
  24. timestamp_change_seconds

Target distribution:
  Class 0 (benign): 372,826 (99.923%)
  Class 1 (suspicious): 288 (0.077%)
  Class imbalance ratio: 1:1295


In [308]:
# Cell 5: Handle Missing Values

print("\n" + "="*80)
print("HANDLING MISSING VALUES")
print("="*80)

print("\nMissing values before:")
missing_before = X.isnull().sum()
print(f"  Columns with missing values: {(missing_before > 0).sum()}")
if (missing_before > 0).any():
    print("\nTop columns with missing values:")
    print(missing_before[missing_before > 0].sort_values(ascending=False).head(10))

# Fill missing values
# Boolean columns: Fill with False
bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].fillna(False)

# Datetime columns: Convert to numeric (days since epoch) or drop
datetime_cols = X.select_dtypes(include='datetime64').columns
for col in datetime_cols:
    if X[col].notna().sum() > 100:  # Keep if enough non-null values
        X[col] = (X[col] - pd.Timestamp('1970-01-01')).dt.total_seconds() / 86400  # Days since epoch
        X[col] = X[col].fillna(X[col].median())
    else:
        X = X.drop(columns=[col])
        print(f"  Dropped datetime column (too many nulls): {col}")

# Numeric columns: Fill with median
numeric_cols = X.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())

print("\nMissing values after:")
missing_after = X.isnull().sum()
print(f"  Columns with missing values: {(missing_after > 0).sum()}")
print(f"  Total missing values: {missing_after.sum()}")

print(f"\nFinal feature matrix shape: {X.shape}")



HANDLING MISSING VALUES

Missing values before:
  Columns with missing values: 1

Top columns with missing values:
timestamp_change_seconds    370400
dtype: int64

Missing values after:
  Columns with missing values: 0
  Total missing values: 0

Final feature matrix shape: (373114, 24)


In [309]:
# Cell 6: Train/Test Split

print("\n" + "="*80)
print("TRAIN/TEST SPLIT (CASE-BASED)")
print("="*80)

# Get unique datasets
unique_datasets = dataset_info.unique()
print(f"\nTotal datasets: {len(unique_datasets)}")
print(f"Datasets: {sorted(unique_datasets)}")

# Randomly select 4 datasets for testing
np.random.seed(42)  # For reproducibility
test_datasets = np.random.choice(unique_datasets, size=4, replace=False)
train_datasets = [d for d in unique_datasets if d not in test_datasets]

print(f"\nTrain datasets ({len(train_datasets)}): {sorted(train_datasets)}")
print(f"Test datasets ({len(test_datasets)}): {sorted(test_datasets)}")

# Create train/test masks
train_mask = dataset_info.isin(train_datasets)
test_mask = dataset_info.isin(test_datasets)

# Split data
X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print(f"\nTrain set:")
print(f"  Total events: {len(X_train):,}")
print(f"  Suspicious: {y_train.sum():,} ({y_train.mean()*100:.3f}%)")
print(f"  Benign: {(~y_train.astype(bool)).sum():,}")

print(f"\nTest set:")
print(f"  Total events: {len(X_test):,}")
print(f"  Suspicious: {y_test.sum():,} ({y_test.mean()*100:.3f}%)")
print(f"  Benign: {(~y_test.astype(bool)).sum():,}")

print(f"\nClass imbalance in train: 1:{(y_train==0).sum()/(y_train==1).sum():.0f}")
print(f"Class imbalance in test: 1:{(y_test==0).sum()/(y_test==1).sum():.0f}")



TRAIN/TEST SPLIT (CASE-BASED)

Total datasets: 22
Datasets: ['01-APT17', '01-PE', '02-PE', '03-APT21', '03-PE', '04-APT28', '04-PE', '05-APT29', '05-PE', '06-APT30', '06-PE', '07-APT37', '07-PE', '08-APT38', '08-PE', '09-PE', '10-DarkHotel663', '10-PE', '11-DarkHotelbbd', '11-PE', '12-PE', '14-Winnti43b']

Train datasets (18): ['02-PE', '03-APT21', '03-PE', '04-APT28', '04-PE', '05-APT29', '06-APT30', '06-PE', '07-APT37', '07-PE', '08-PE', '09-PE', '10-DarkHotel663', '10-PE', '11-DarkHotelbbd', '11-PE', '12-PE', '14-Winnti43b']
Test datasets (4): ['01-APT17', '01-PE', '05-PE', '08-APT38']

Train set:
  Total events: 296,404
  Suspicious: 281 (0.095%)
  Benign: 296,123

Test set:
  Total events: 76,710
  Suspicious: 7 (0.009%)
  Benign: 76,703

Class imbalance in train: 1:1054
Class imbalance in test: 1:10958


## Handle Class Imbalance with SMOTE

**Problem**: Severe class imbalance (0.077% suspicious events)

**Solution**: SMOTE (Synthetic Minority Over-sampling Technique)
- Creates synthetic samples of the minority class
- Balances the training set to improve model learning
- Only applied to training set (NOT test set!)

**Target balance**: Oversample minority class to 10% of majority class (more conservative than 50/50)


In [310]:
# Cell 7: Apply SMOTE

print("\n" + "="*80)
print("APPLYING SMOTE (CLASS IMBALANCE HANDLING)")
print("="*80)

print(f"\nBefore SMOTE:")
print(f"  Train size: {len(X_train):,}")
print(f"  Suspicious: {y_train.sum():,}")
print(f"  Benign: {(y_train==0).sum():,}")
print(f"  Ratio: 1:{(y_train==0).sum()/(y_train==1).sum():.0f}")

# Apply SMOTE with conservative ratio (10% minority)
# This prevents over-generation of synthetic samples
sampling_strategy = 0.10  # Minority will be 10% of majority

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Train size: {len(X_train_balanced):,}")
print(f"  Suspicious: {y_train_balanced.sum():,}")
print(f"  Benign: {(y_train_balanced==0).sum():,}")
print(f"  Ratio: 1:{(y_train_balanced==0).sum()/(y_train_balanced==1).sum():.0f}")

print(f"\nSynthetic samples created: {len(X_train_balanced) - len(X_train):,}")



APPLYING SMOTE (CLASS IMBALANCE HANDLING)

Before SMOTE:
  Train size: 296,404
  Suspicious: 281
  Benign: 296,123
  Ratio: 1:1054

After SMOTE:
  Train size: 325,735
  Suspicious: 29,612
  Benign: 296,123
  Ratio: 1:10

Synthetic samples created: 29,331


## Model Training

Train 4 different models and compare performance:

### 1. Random Forest
- **Pros**: Robust, handles non-linear relationships, provides feature importance
- **Hyperparameters**: 100 trees, max depth 20, class weights

### 2. XGBoost  
- **Pros**: State-of-the-art gradient boosting, handles imbalance well
- **Hyperparameters**: 100 estimators, learning rate 0.1, max depth 6

### 3. LightGBM
- **Pros**: Fast training, efficient memory usage
- **Hyperparameters**: 100 estimators, learning rate 0.1, max depth 6

### 4. Logistic Regression
- **Pros**: Simple, interpretable, fast
- **Hyperparameters**: L2 regularization, class weights


In [311]:
# Cell 8: Train Models

print("\n" + "="*80)
print("MODEL TRAINING")
print("="*80)

# Calculate class weights for models that support it
class_weight_ratio = (y_train_balanced==0).sum() / (y_train_balanced==1).sum()
class_weights = {0: 1, 1: class_weight_ratio}

models = {}
results = []

# 1. RANDOM FOREST
print("\n" + "-"*80)
print("1. RANDOM FOREST")
print("-"*80)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

print("Training Random Forest...")
rf_model.fit(X_train_balanced, y_train_balanced)

# Evaluate on test set
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_pred_proba_rf)

print(f"  Precision: {rf_precision:.3f}")
print(f"  Recall: {rf_recall:.3f}")
print(f"  F1-Score: {rf_f1:.3f}")
print(f"  ROC-AUC: {rf_auc:.3f}")

models['Random Forest'] = rf_model
results.append({
    'Model': 'Random Forest',
    'Precision': rf_precision,
    'Recall': rf_recall,
    'F1-Score': rf_f1,
    'ROC-AUC': rf_auc
})

# 2. XGBOOST
print("\n" + "-"*80)
print("2. XGBOOST")
print("-"*80)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=class_weight_ratio,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print("Training XGBoost...")
xgb_model.fit(X_train_balanced, y_train_balanced, verbose=False)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"  Precision: {xgb_precision:.3f}")
print(f"  Recall: {xgb_recall:.3f}")
print(f"  F1-Score: {xgb_f1:.3f}")
print(f"  ROC-AUC: {xgb_auc:.3f}")

models['XGBoost'] = xgb_model
results.append({
    'Model': 'XGBoost',
    'Precision': xgb_precision,
    'Recall': xgb_recall,
    'F1-Score': xgb_f1,
    'ROC-AUC': xgb_auc
})

# 3. LIGHTGBM
print("\n" + "-"*80)
print("3. LIGHTGBM")
print("-"*80)

lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("Training LightGBM...")
lgb_model.fit(X_train_balanced, y_train_balanced)

y_pred_lgb = lgb_model.predict(X_test)
y_pred_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

lgb_precision = precision_score(y_test, y_pred_lgb)
lgb_recall = recall_score(y_test, y_pred_lgb)
lgb_f1 = f1_score(y_test, y_pred_lgb)
lgb_auc = roc_auc_score(y_test, y_pred_proba_lgb)

print(f"  Precision: {lgb_precision:.3f}")
print(f"  Recall: {lgb_recall:.3f}")
print(f"  F1-Score: {lgb_f1:.3f}")
print(f"  ROC-AUC: {lgb_auc:.3f}")

models['LightGBM'] = lgb_model
results.append({
    'Model': 'LightGBM',
    'Precision': lgb_precision,
    'Recall': lgb_recall,
    'F1-Score': lgb_f1,
    'ROC-AUC': lgb_auc
})

# 4. LOGISTIC REGRESSION
print("\n" + "-"*80)
print("4. LOGISTIC REGRESSION")
print("-"*80)

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight=class_weights,
    random_state=42,
    max_iter=1000,
    n_jobs=-1
)

print("Training Logistic Regression...")
lr_model.fit(X_train_scaled, y_train_balanced)

y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_pred_proba_lr)

print(f"  Precision: {lr_precision:.3f}")
print(f"  Recall: {lr_recall:.3f}")
print(f"  F1-Score: {lr_f1:.3f}")
print(f"  ROC-AUC: {lr_auc:.3f}")

models['Logistic Regression'] = (lr_model, scaler)  # Save both model and scaler
results.append({
    'Model': 'Logistic Regression',
    'Precision': lr_precision,
    'Recall': lr_recall,
    'F1-Score': lr_f1,
    'ROC-AUC': lr_auc
})



MODEL TRAINING

--------------------------------------------------------------------------------
1. RANDOM FOREST
--------------------------------------------------------------------------------
Training Random Forest...
  Precision: 0.021
  Recall: 0.857
  F1-Score: 0.041
  ROC-AUC: 0.998

--------------------------------------------------------------------------------
2. XGBOOST
--------------------------------------------------------------------------------
Training XGBoost...
  Precision: 0.023
  Recall: 1.000
  F1-Score: 0.044
  ROC-AUC: 0.999

--------------------------------------------------------------------------------
3. LIGHTGBM
--------------------------------------------------------------------------------
Training LightGBM...
  Precision: 0.024
  Recall: 1.000
  F1-Score: 0.047
  ROC-AUC: 0.999

--------------------------------------------------------------------------------
4. LOGISTIC REGRESSION
-------------------------------------------------------------------------

In [312]:
# Cell 9: Model Comparison

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

# Create comparison DataFrame
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("\nModel Performance on Test Set:")
print(comparison_df.to_string(index=False))

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1-Score']

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name}")
print(f"F1-Score: {best_f1:.3f}")
print(f"{'='*80}")

# Get best model
if best_model_name == 'Logistic Regression':
    best_model, best_scaler = models[best_model_name]
else:
    best_model = models[best_model_name]
    best_scaler = None

# Save comparison
comparison_path = OUTPUT_DIR / 'model_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f"\nModel comparison saved: {comparison_path}")



MODEL COMPARISON

Model Performance on Test Set:
              Model  Precision   Recall  F1-Score  ROC-AUC
           LightGBM   0.024055 1.000000  0.046980 0.998885
            XGBoost   0.022581 1.000000  0.044164 0.998880
      Random Forest   0.020906 0.857143  0.040816 0.998097
Logistic Regression   0.004894 0.857143  0.009732 0.994211

BEST MODEL: LightGBM
F1-Score: 0.047

Model comparison saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/model_comparison.csv


## Feature Importance Analysis

Analyze which features are most important for the best model.

**Note**: Feature importance interpretation depends on model type:
- **Tree-based models** (RF, XGBoost, LightGBM): Gini importance or gain
- **Logistic Regression**: Coefficient absolute values


In [313]:
# Cell 10: Feature Importance

print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

if best_model_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    # Get feature importance from tree-based model
    if best_model_name == 'Random Forest':
        importance = best_model.feature_importances_
    elif best_model_name == 'XGBoost':
        importance = best_model.feature_importances_
    else:  # LightGBM
        importance = best_model.feature_importances_
    
    # Create DataFrame
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
elif best_model_name == 'Logistic Regression':
    # Get coefficient absolute values
    importance = np.abs(best_model.coef_[0])
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': importance
    }).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Save feature importance
feature_importance_path = OUTPUT_DIR / 'feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False)
print(f"\nFeature importance saved: {feature_importance_path}")



FEATURE IMPORTANCE ANALYSIS

Top 20 Most Important Features:
                        feature  importance
                filename_length         809
          events_in_1min_window         494
           event_count_per_file         448
                  is_executable         252
          events_in_5min_window         215
       timestamp_change_seconds         145
            time_reversal_event         139
                 is_system_file         135
             basic_info_changed          95
cross_artifact_validation_score          77
               zero_in_creation          33
            zero_in_nanoseconds          28
           has_logfile_evidence          21
                       is_image          21
               zero_in_accessed          20
          update_resident_value          13
               zero_in_modified           9
           zero_in_mft_modified           2
                    is_document           1
                     is_archive           1

Feature impor

## Save Best Model

Save the best model for production use in Phase 4 (Validation) and Phase 5 (Autopsy integration).

**Saved files**:
- `models/best_model.pkl`: Trained model
- `models/feature_scaler.pkl`: Feature scaler (if Logistic Regression)
- `models/feature_columns.pkl`: List of feature columns (for consistency)


In [314]:
# Cell 11: Save Best Model

print("\n" + "="*80)
print("SAVING BEST MODEL")
print("="*80)

# Save best model
model_path = MODELS_DIR / 'best_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nBest model saved: {model_path}")
print(f"Model type: {best_model_name}")

# Save scaler if Logistic Regression
if best_scaler is not None:
    scaler_path = MODELS_DIR / 'feature_scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(best_scaler, f)
    print(f"Feature scaler saved: {scaler_path}")

# Save feature columns
feature_cols_path = MODELS_DIR / 'feature_columns.pkl'
with open(feature_cols_path, 'wb') as f:
    pickle.dump(list(X.columns), f)
print(f"Feature columns saved: {feature_cols_path}")

# Save model metadata
metadata = {
    'model_name': best_model_name,
    'f1_score': best_f1,
    'precision': comparison_df.iloc[0]['Precision'],
    'recall': comparison_df.iloc[0]['Recall'],
    'roc_auc': comparison_df.iloc[0]['ROC-AUC'],
    'train_size': len(X_train_balanced),
    'test_size': len(X_test),
    'num_features': len(X.columns),
    'smote_ratio': sampling_strategy
}

metadata_path = MODELS_DIR / 'model_metadata.pkl'
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"Model metadata saved: {metadata_path}")



SAVING BEST MODEL

Best model saved: /Users/soni/Github/Digital-Detectives_Thesis/models/best_model.pkl
Model type: LightGBM
Feature columns saved: /Users/soni/Github/Digital-Detectives_Thesis/models/feature_columns.pkl
Model metadata saved: /Users/soni/Github/Digital-Detectives_Thesis/models/model_metadata.pkl


## Test Set Detailed Evaluation

Analyze the best model's performance on the test set in detail.

**Metrics**:
- Confusion Matrix
- Classification Report
- Per-dataset performance


In [315]:
# Cell 12: Test Set Evaluation

print("\n" + "="*80)
print("TEST SET DETAILED EVALUATION")
print("="*80)

# Get predictions from best model
if best_model_name == 'Logistic Regression':
    y_pred_best = best_model.predict(X_test_scaled)
    y_pred_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    y_pred_best = best_model.predict(X_test)
    y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(f"                 Predicted Negative  Predicted Positive")
print(f"Actual Negative      {cm[0,0]:8d}           {cm[0,1]:8d}")
print(f"Actual Positive      {cm[1,0]:8d}           {cm[1,1]:8d}")

print(f"\nTrue Negatives (TN): {cm[0,0]:,}")
print(f"False Positives (FP): {cm[0,1]:,}")
print(f"False Negatives (FN): {cm[1,0]:,}")
print(f"True Positives (TP): {cm[1,1]:,}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Benign', 'Suspicious']))

# Save test predictions
test_results = pd.DataFrame({
    'dataset': dataset_info[test_mask].values,
    'true_label': y_test.values,
    'predicted_label': y_pred_best,
    'prediction_probability': y_pred_proba_best
})

test_results_path = OUTPUT_DIR / 'test_results.csv'
test_results.to_csv(test_results_path, index=False)
print(f"\nTest results saved: {test_results_path}")

# Per-dataset performance
print("\nPer-Dataset Performance on Test Set:")
for dataset in sorted(test_results['dataset'].unique()):
    dataset_mask = test_results['dataset'] == dataset
    dataset_true = test_results.loc[dataset_mask, 'true_label']
    dataset_pred = test_results.loc[dataset_mask, 'predicted_label']
    
    if dataset_true.sum() > 0:  # If there are suspicious events
        dataset_recall = recall_score(dataset_true, dataset_pred)
        dataset_precision = precision_score(dataset_true, dataset_pred) if dataset_pred.sum() > 0 else 0
        print(f"  {dataset:20s} Recall: {dataset_recall:.3f} | Precision: {dataset_precision:.3f} | Suspicious: {dataset_true.sum()}")
    else:
        print(f"  {dataset:20s} No suspicious events")



TEST SET DETAILED EVALUATION

Confusion Matrix:
                 Predicted Negative  Predicted Positive
Actual Negative         76419                284
Actual Positive             0                  7

True Negatives (TN): 76,419
False Positives (FP): 284
False Negatives (FN): 0
True Positives (TP): 7

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     76703
  Suspicious       0.02      1.00      0.05         7

    accuracy                           1.00     76710
   macro avg       0.51      1.00      0.52     76710
weighted avg       1.00      1.00      1.00     76710


Test results saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/test_results.csv

Per-Dataset Performance on Test Set:
  01-APT17             Recall: 1.000 | Precision: 0.667 | Suspicious: 2
  01-PE                Recall: 1.000 | Precision: 0.027 | Suspicious: 2
  05-PE                Recall: 1.000 | 

## Validation Set Predictions (Final Test)

Test the best model on the 5 completely unseen validation datasets.

**Critical**: These datasets were NOT used in training or testing. This is the true measure of model generalization.

**Validation datasets**:
1. LoneWolf (12 timestomped files)
2. 02-APT19 (1 file)
3. 09-APT40 (1 file)
4. 12-Kimsuky (3 files)
5. 13-Winnti731 (1 file)

**Total**: 18 timestomped files

**Note**: We have NO ground truth labels in these CSVs. We'll save predictions and compare with ground truth in Phase 4.


In [316]:
# Cell 13: Validation Set Predictions

print("\n" + "="*80)
print("VALIDATION SET PREDICTIONS")
print("="*80)

validation_datasets = ['LoneWolf', '02-APT19', '09-APT40', '12-Kimsuky', '13-Winnti731']

all_validation_predictions = []

for dataset_name in validation_datasets:
    try:
        print(f"\n{'-'*80}")
        print(f"Processing: {dataset_name}")
        print(f"{'-'*80}")
        
        # Load validation features
        val_features_path = VALIDATION_FEATURES_DIR / f'{dataset_name}_features.csv'
        val_df = pd.read_csv(val_features_path, low_memory=False)
        
        print(f"  Loaded: {len(val_df):,} events")
        
        # Extract same features used in training
        val_X = val_df[X.columns]  # Use same columns as training
        
        # Handle missing values same way as training
        for col in bool_cols:
            if col in val_X.columns:
                val_X[col] = val_X[col].fillna(False)
        
        for col in val_X.select_dtypes(include=[np.number]).columns:
            if val_X[col].isnull().any():
                val_X[col] = val_X[col].fillna(val_X[col].median())
        
        # Make predictions
        if best_model_name == 'Logistic Regression':
            val_X_scaled = best_scaler.transform(val_X)
            val_predictions = best_model.predict(val_X_scaled)
            val_probabilities = best_model.predict_proba(val_X_scaled)[:, 1]
        else:
            val_predictions = best_model.predict(val_X)
            val_probabilities = best_model.predict_proba(val_X)[:, 1]
        
        # Count suspicious predictions
        suspicious_count = val_predictions.sum()
        suspicious_pct = (val_predictions.mean() * 100)
        
        print(f"  Predictions:")
        print(f"    Suspicious events: {suspicious_count:,} ({suspicious_pct:.2f}%)")
        print(f"    Benign events: {(~val_predictions.astype(bool)).sum():,}")
        print(f"    Mean probability: {val_probabilities.mean():.4f}")
        print(f"    Max probability: {val_probabilities.max():.4f}")
        
        # Identify unique files flagged as suspicious
        if 'filename' in val_df.columns:
            suspicious_mask = val_predictions.astype(bool)
            suspicious_files = val_df.loc[suspicious_mask, 'filename'].unique()
            print(f"    Unique files flagged: {len(suspicious_files)}")
            if len(suspicious_files) > 0 and len(suspicious_files) <= 20:
                print(f"    Files: {', '.join(suspicious_files)}")
        
        # Save predictions with original data
        val_results = val_df.copy()
        val_results['predicted_label'] = val_predictions
        val_results['prediction_probability'] = val_probabilities
        val_results['dataset'] = dataset_name
        
        all_validation_predictions.append(val_results)
        
        # Save individual dataset predictions
        val_output_path = OUTPUT_DIR / f'{dataset_name}_predictions.csv'
        val_results.to_csv(val_output_path, index=False)
        print(f"  Saved: {val_output_path}")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()
        continue

# Combine all validation predictions
if all_validation_predictions:
    combined_validation = pd.concat(all_validation_predictions, ignore_index=True)
    combined_path = OUTPUT_DIR / 'all_validation_predictions.csv'
    combined_validation.to_csv(combined_path, index=False)
    print(f"\nAll validation predictions saved: {combined_path}")



VALIDATION SET PREDICTIONS

--------------------------------------------------------------------------------
Processing: LoneWolf
--------------------------------------------------------------------------------
  Loaded: 34,300 events
  Predictions:
    Suspicious events: 0 (0.00%)
    Benign events: 34,300
    Mean probability: 0.0002
    Max probability: 0.1458
    Unique files flagged: 0
  Saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/LoneWolf_predictions.csv

--------------------------------------------------------------------------------
Processing: 02-APT19
--------------------------------------------------------------------------------
  Loaded: 23,803 events
  Predictions:
    Suspicious events: 0 (0.00%)
    Benign events: 23,803
    Mean probability: 0.0005
    Max probability: 0.3575
    Unique files flagged: 0
  Saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/02-APT19_predictions.csv


## Phase 3 Summary

Review all outputs and prepare for Phase 4.


In [317]:
# Cell 14: Phase 3 Summary

print("\n" + "="*80)
print("PHASE 3 COMPLETE - SUMMARY")
print("="*80)

print(f"\nBest Model: {best_model_name}")
print(f"Test Set Performance:")
print(f"  Precision: {comparison_df.iloc[0]['Precision']:.3f}")
print(f"  Recall: {comparison_df.iloc[0]['Recall']:.3f}")
print(f"  F1-Score: {comparison_df.iloc[0]['F1-Score']:.3f}")
print(f"  ROC-AUC: {comparison_df.iloc[0]['ROC-AUC']:.3f}")

print(f"\nTraining Details:")
print(f"  Train datasets: {len(train_datasets)}")
print(f"  Test datasets: {len(test_datasets)}")
print(f"  Train size (after SMOTE): {len(X_train_balanced):,}")
print(f"  Test size: {len(X_test):,}")
print(f"  Features used: {len(X.columns)}")

print(f"\nValidation Predictions:")
print(f"  Datasets processed: {len(validation_datasets)}")
print(f"  Total validation events: {sum(len(vp) for vp in all_validation_predictions):,}")

print(f"\nOutput Files:")
print(f"  Model: {MODELS_DIR / 'best_model.pkl'}")
print(f"  Model comparison: {OUTPUT_DIR / 'model_comparison.csv'}")
print(f"  Feature importance: {OUTPUT_DIR / 'feature_importance.csv'}")
print(f"  Test results: {OUTPUT_DIR / 'test_results.csv'}")
print(f"  Validation predictions: {OUTPUT_DIR / 'all_validation_predictions.csv'}")

print("\n" + "="*80)
print("NEXT: Phase 4 - Compare validation predictions with ground truth")
print("="*80)



PHASE 3 COMPLETE - SUMMARY

Best Model: LightGBM
Test Set Performance:
  Precision: 0.024
  Recall: 1.000
  F1-Score: 0.047
  ROC-AUC: 0.999

Training Details:
  Train datasets: 18
  Test datasets: 4
  Train size (after SMOTE): 325,735
  Test size: 76,710
  Features used: 24

Validation Predictions:
  Datasets processed: 5
  Total validation events: 113,325

Output Files:
  Model: /Users/soni/Github/Digital-Detectives_Thesis/models/best_model.pkl
  Model comparison: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/model_comparison.csv
  Feature importance: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/feature_importance.csv
  Test results: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/test_results.csv
  Validation predictions: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/all_validation_predictions.csv

NEXT: Phase 4 - Compare validat